<a href="https://colab.research.google.com/github/Tmiller68/machine-learning-fundamentals/blob/main/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded=files.upload()

Saving adult.data to adult.data


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier
!pip install xgboost

column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv("adult.data", names=column_names, header=None)

df = df.replace(" ?", np.nan)
df = df.dropna()

X = df.drop("income", axis=1)
Y = df["income"]

Y = Y.str.strip()

X = pd.get_dummies(X, drop_first=True, dtype=int)

label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

unlimited_tree = DecisionTreeClassifier(random_state=42)
unlimited_tree.fit(X_train, Y_train)

limited_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
limited_tree.fit(X_train, Y_train)

unlimited_train_predictions = unlimited_tree.predict(X_train)
unlimited_test_predictions = unlimited_tree.predict(X_test)

limited_train_predictions = limited_tree.predict(X_train)
limited_test_predictions = limited_tree.predict(X_test)

unlimited_train_accuracy = accuracy_score(
    Y_train,
    unlimited_train_predictions
)

unlimited_test_accuracy = accuracy_score(
    Y_test,
    unlimited_test_predictions
)

limited_train_accuracy = accuracy_score(
    Y_train,
    limited_train_predictions
)

limited_test_accuracy = accuracy_score(
    Y_test,
    limited_test_predictions
)

print("Unlimited Decision Tree")
print("Training accuracy:", unlimited_train_accuracy)
print("Test accuracy:", unlimited_test_accuracy)

print("Decision Tree with max_depth=5")
print("Training accuracy:", limited_train_accuracy)
print("Test accuracy:", limited_test_accuracy)

Unlimited Decision Tree
Training accuracy: 1.0
Test accuracy: 0.8140228741919443
Decision Tree with max_depth=5
Training accuracy: 0.8429275974967881
Test accuracy: 0.8408751864743909


In [16]:
logistic_model=Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])
tree_model=DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
logistic_scores=cross_val_score(
    logistic_model,
    X,
    Y,
    cv=5,
    scoring="accuracy"
)
tree_scores=cross_val_score(
    tree_model,
    X,
    Y,
    cv=5,
    scoring="accuracy"
)

In [17]:
logistic_model.fit(X,Y)
coefficients=logistic_model.named_steps["logistic"].coef_[0]
coefficient_table=pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
})
coefficient_table=coefficient_table.sort_values(
    by="Coefficient",
    ascending=False
)
coefficient_table.head(10)
coefficient_table.tail(10)

,Feature,Coefficient
6,workclass_ Local-gov,-0.176635
80,native_country_ Mexico,-0.197297
93,native_country_ United-States,-0.203797
7,workclass_ Private,-0.222185
30,marital_status_ Never-married,-0.229579
39,occupation_ Other-service,-0.255398
48,relationship_ Own-child,-0.266780
9,workclass_ Self-emp-not-inc,-0.274914
40,occupation_ Priv-house-serv,-0.284082
24,education_ Preschool,-0.607853


In [18]:
print(
    "Logistic Regression:",
    round(logistic_scores.mean(), 4),
    "±",
    round(logistic_scores.std(), 4)
)

print(
    "Decision Tree:",
    round(tree_scores.mean(), 4),
    "±",
    round(tree_scores.std(), 4)
)
print("Top positive coefficients: ")
print(coefficient_table.head(10))
print("\nTop negative coefficients:")
print(coefficient_table.tail(10))

Logistic Regression: 0.8476 ± 0.004
Decision Tree: 0.8427 ± 0.0052
Top positive coefficients: 
                               Feature  Coefficient
3                         capital_gain     2.373122
28  marital_status_ Married-civ-spouse     1.031989
2                        education_num     0.520602
55                           sex_ Male     0.404321
5                       hours_per_week     0.352280
0                                  age     0.333518
50                  relationship_ Wife     0.284858
35         occupation_ Exec-managerial     0.273448
4                         capital_loss     0.259199
54                         race_ White     0.211842

Top negative coefficients:
                          Feature  Coefficient
6            workclass_ Local-gov    -0.176635
80         native_country_ Mexico    -0.197297
93  native_country_ United-States    -0.203797
7              workclass_ Private    -0.222185
30  marital_status_ Never-married    -0.229579
39      occupation_ Oth

In [20]:
random_forest_model=RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)
xgboost_model=XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [21]:
logistic_results=cross_validate(
    logistic_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
    )
tree_results=cross_validate(
    tree_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [22]:
random_forest_results = cross_validate(
    random_forest_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

xgboost_results = cross_validate(
    xgboost_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [26]:
result["test_score"]
result["fit_time"]

NameError: name 'result' is not defined

In [28]:
result_table=pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "CV Accuray": [
        logistic_results["test_score"].mean(),
        tree_results["test_score"].mean(),
        random_forest_results["test_score"].mean(),
        xgboost_results["test_score"].mean()
    ],
    "Accuracy STD": [
        logistic_results["test_score"].std(),
        tree_results["test_score"].std(),
        random_forest_results["test_score"].std(),
        xgboost_results["test_score"].std()
    ],
    "Mean Fit Time": [
        logistic_results["fit_time"].mean(),
        tree_results["fit_time"].mean(),
        random_forest_results["fit_time"].mean(),
        xgboost_results["fit_time"].mean(),
    ]
})
result_table

,Model,CV Accuray,Accuracy STD,Mean Fit Time
0,Logistic Regression,0.847557,0.004027,2.108432
1,Decision Tree,0.842750,0.005201,0.325819
2,Random Forest,0.835986,0.003215,2.259282
3,XGBoost,0.868378,0.003757,1.184986
